# Download the dataset


# Confident Legal Research Application

- Main stakeholder/consumer: Junior Legal Assistant
- Distributed: No


**Goal:Improve quality & speed of case research for junior assistants**

## Baseline System

To create evaluation process we would need to have some baseline system in place. That is why we are creating a **Naive implementation** with the following initial setup:

- One sparse vector embedding (simplest 384 encoder)
- Search function with no reranker
- Naive RAG pipeline

In [2]:
%load_ext autoreload
%autoreload 2

### Initialize dataset

We will use famouse CLERC dataset for this tutorial. Two reasons:
1. There exist queries in the dataset that we can use as a baseline for complex law queries 
2. The combination of queries/positive/negative responses provides a good baseline for the *golden set* that could be used for testing

In [3]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [4]:
sample_tuple = clerc_data.sample()

sample_tuple

(Query(query_id='355739', query='the time vision was obscured by smoke and fumes created in part by his landbased lack of care. We conclude that § 1333(a)(2) of the Lands Act, when read against the background of legislative history outlined in Rodrigue and the increasing judicial and legislative dissatisfaction with strict locality as the sole test of jurisdiction, precludes us from treating Monk’s claim against C & K and Chapman as a suit in admiralty. Accordingly the Texas doctrine of contributory negligence bars recovery by Monk’s heirs from C & K and Chapman. Dearborn has not claimed the benefit of the Texas contributory negligence bar, and correctly so. Suits by landsmen injured aboard ship in navigable waters have traditionally been within maritime jurisdiction.  REDACTED Moreover, Dearborn’s situation is different from that of the platform defendants whose breaches of duty were platform-related, while Dearborn’s concurring breaches were admiralty-related. As between platform def

### Initialize vectors 

- use `qdrant_client` for storing vectors
- use `fastembed` for the simplest embeddings generation pipelines


We will also store the embeddings locally and try to handle them paralelly 

>NB!: This notebook is optimized to run on macbook devices. Try to adapt embedding config to your specific hardware

In [5]:
from qdrant_client import QdrantClient
import os

qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

In [6]:
# constants
COLLECTION_NAME = 'lawyer_citations'
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SIZE = 384

In [7]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance


config = EmbeddingConfig(
    name='dense_base',
    model_id=DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
    providers=["CoreMLExecutionProvider", "CPUExecutionProvider"],
    parallel=None,
)

In [8]:
from src.indexing import EmbeddingCache
embedding_cache = EmbeddingCache()

In [9]:
from src.indexing import DocumentIndexer

document_vectors = DocumentIndexer(
    client,
    COLLECTION_NAME,
    embeddings=[config],
    cache=embedding_cache
)

In [10]:
document_vectors.ensure_collection()

In [20]:
document_vectors.upload(items=list(clerc_data.corpus.values()), batch_size=128)

embed:dense_base:   0%|          | 0/41167 [00:00<?, ?it/s]

In [11]:
client.get_collection(COLLECTION_NAME).points_count, (client.get_collection(COLLECTION_NAME).points_count or 0) > 0

(41167, True)

### Create simple semantic search and RAG pipelines

- `DenseSearcher` is the simplest implementation of the semantic search: no sparse vectors, multivectors, no reranking. Just query API
- `LegalAssistant` - uses `DenseSearcher` for the retrieval and generates the results based on that. We use `anthropic api` with `lite-llm`

In [12]:
from src.search import DenseSearcher
from src.rag import LegalAssistant

qa = clerc_data.sample()
searcher = DenseSearcher(client, COLLECTION_NAME, config)
legal_assistant = LegalAssistant(searcher)


In [17]:
result = searcher.search(qa[0].query)
result[0].text, qa[1].text

('F.2d 1214, 1216 (9th Cir.1978), the Ninth Circuit incorporated the criminal provisions of section 1681q into sections 1681n [Civil liability for willful noncompliance] and 1681o [Civil liability. for negligent noncompliance] and held that a user who obtains a consumer report under false pretenses, as those words are used in section 1681q, can be civilly liable under either section 1681n or 1681o. In Hansen, Judge Carter wrote that “[t]he standard for determining when a consumer report has been obtained under false pretenses will usually be defined in relation to the permissible purposes of consumer reports which are enumerated in 15 U.S.C. § 1681b,” id. at 1219 (footnote omitted), “that obtaining a consumer report ... without disclosing the impermissible purpose for which the report is desired can constitute obtaining consumer information under false pretenses.” Id. at 971-72. Thus, whether defendants are users or consumer reporting agencies, they could only obtain Korotki’s report i

In [18]:
answer = legal_assistant.ask("My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems")

print(answer)

Based on the case excerpts provided, none of them directly address the specific legal issue of a landlord seeking payment for property damage (a broken wall) in the absence of a signed hand-in (or check-out/handover) protocol. The excerpts cover unrelated legal matters, including:

1. A bankruptcy dispute over post-petition claims for furniture repossession costs (doc_id: 9457344).
2. A challenge to an administrative regulation regarding assignment of farm rent payments (doc_id: 14718586).
3. A government contract dispute over equitable adjustments (doc_id: 8705202).
4. A landlord-tenant dispute under the Emergency Price Control Act concerning rent overcharges — not property damage or handover protocols (doc_id: 3706151).
5. A bankruptcy/receivership jurisdiction matter (doc_id: 4359176).

The only excerpt that involves a landlord-tenant relationship at all is doc_id: 3706151, which concerns a landlord allegedly charging rent above the legally permitted maximum under wartime rent contr

### Basis of the evaluation framework

Even on a single query, the retrieval layer is visibly underperforming — by design. The PoC intentionally omits sparse vectors, reranking, and query rewriting so the failure modes those techniques usually hide surface up front.

What saved this run is the LLM: `claude-sonnet-4.6` is strong enough to notice the retrieved excerpts don't answer the question and refuse to fabricate. That safety net is fragile. With a weaker model, an opaque pipeline, or domain experts who can't audit the citation chain, the same retrieval bug becomes invisible — and the system confidently misleads a junior associate who can't yet tell the difference.

## Goal oriented evaluation specification

The evaluation is designed **top-down**: a system that doesn't serve its business goal isn't worth measuring at the component level.

### Inputs/Assumptions

The preconditions feeding into the methodology:
- **GOAL:** Confident Legal Research
- **CONSUMER:** Junior associate — limited domain depth, needs guidance and explanations
- **DISTRIBUTED:** No — single goal, single consumer

In [33]:
from IPython.display import HTML

HTML("""
<div style="text-align: center;">
    <img src="diagrams/eval-inputs.svg" width="300">
</div>
""")

### Goal Decomposition

**Goal decomposition** bridges the abstract business goal and the concrete metrics that measure it. Before we can quantify, we have to be specific about *what the system is* and *what it must do*.

Two questions to anchor the decomposition:
- **Is this a *business* goal?** Yes — it describes the system's *utility*, not its internals.
- **Can we make *component-level* assumptions?** Yes — the system has two: **searcher** (retrieval, with well-understood mechanics) and **rag** (LLM-driven generation).

Refined business objective:

`Improve quality & speed of case research for junior associates`

Technical commitments already baked into the codebase:

`Confident & fast legal agentic search` *(too vague to test as-is — decomposition follows)*

Decomposition is one of the oldest design principles in software engineering. Here we use it to turn the vague tech goal into the evaluation context:

![System Goal Decomposition](diagrams/system-goal-decomposition.svg)

At the system level, three NFRs apply to every sub-system:
- **Trustworthiness** — can the consumer rely on the output without verification, and fail loudly when it can't?
- **Correctness** — does the output match ground truth for the task it owns?
- **Performance** — is it fast enough to be useful in the consumer's loop?

These stay vague until we map them to individual components. Each sub-system has a different consumer, and the consumer determines what each NFR *operationally means*:

![Component Level Goal Decomposition](diagrams/component-level-goal-decomposition.svg)

**Search** is consumed by RAG (programmatic). Its failure surface is narrow — a bad ranked list, an over-confident empty result, or a slow query. Three flat axes are enough.

**RAG** is consumed by a junior associate (human). Its failure surface is richer — confidently-wrong differs sharply from correctly-refused, and citation grounding, hedging, and clarity are all separately damaging when they fail. So the decomposition goes deeper on the RAG side.

**The depth asymmetry is itself the finding:** it tells the reader where the evaluation effort has to concentrate.

One consequence worth calling out early: dense semantic search has no natural zero — every query embeds *somewhere*, every doc embeds *somewhere*, cosine similarity always produces an ordering. So *no relevant doc in the corpus → empty top-K* isn't emergent behavior; it has to be engineered via a calibrated threshold. That's why **Calibration** sits as the headline trustworthiness bullet on the search side.

Without per-component decomposition, a wrong final answer can't be localized to bad retrieval vs. bad synthesis — and what can't be localized can't be fixed. That's exactly what the current PoC shows: RAG had to reject its own retriever's ranking because there was no earlier signal — no calibrated empty, no per-component metric — that would have told the operator the search side was the problem.

### Ranking Quality as example for Non-functional requirements mapped to the metrics


Next part of the tutorial we will focus only on one non-functional requirement that was obviously needs some improvement by the simple implementation we have. I am talking about **Ranking quality**, a search component NFR (non-functional requirement, will be used from now on).

What we do is walk through a series of decision questions that help identify the most appropriate metric to use in this situation.

![Evaluation Metric Definition](./diagrams/evaluation-path-to-metric.svg)


The end result is that currently have a clear path why we would measure `NDCG@10` - because:
- We have the golden dataset from `CLERC`
- We want to measure what is the current **Ranking Quality** of the system

We simplified two routes here though: 
- There is no validation of the coverage / leakage / drift for the dataset, that is a separate topic within itself
- We assume that for now it will be enough to ignore the consumer being an LLM instead of a person. Technically this can be a mistake however, practically acknowledging that our metric is flawed, however best we can do so far is better than the other way around.

In [19]:
clerc_data

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [25]:
clerc_data.qrels

{'247774': {'22437426'},
 '736597': {'195263'},
 '126954': {'21178393'},
 '806778': {'22093035'},
 '745199': {'10234535'},
 '14115': {'5627904'},
 '840906': {'12238541'},
 '134081': {'1274457'},
 '647086': {'22438326'},
 '369085': {'5011261'},
 '704911': {'3727252'},
 '522851': {'22596202'},
 '24981': {'22799663'},
 '352651': {'4344972'},
 '58924': {'13617648'},
 '85461': {'22360115'},
 '448165': {'22656949'},
 '778940': {'7815522'},
 '418643': {'17004973'},
 '719576': {'22564545'},
 '407383': {'21552799'},
 '854122': {'22259485'},
 '699414': {'22661193'},
 '252125': {'6508378'},
 '397070': {'16959505'},
 '417371': {'23705757'},
 '673872': {'22748467'},
 '238583': {'10938805'},
 '701715': {'21190195'},
 '94329': {'23682717'},
 '440617': {'13747782'},
 '790841': {'8432440'},
 '145475': {'3845113'},
 '696065': {'668079'},
 '536243': {'22439544'},
 '391607': {'22609755'},
 '783482': {'10841433'},
 '746128': {'21948389'},
 '547023': {'12593357'},
 '831733': {'22608269'},
 '356172': {'94822

In [ ]:
i = 1
for query in clerc_data.queries:
    current_query = clerc_data.queries[query]

    # golden answer
    i += 1
    if i > 10:
        break
    golden_answer = clerc_data.qrels[current_query.query_id]

{'22437426'}
{'195263'}
{'21178393'}
{'22093035'}
{'10234535'}
{'5627904'}
{'12238541'}
{'1274457'}
{'22438326'}
